### EDA On Ridership and Sentiment

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "vscode"

In [2]:
#csv for eda, api endpoint later for more interactive visualizations
sentiment = pd.read_csv("data/MTA_Survey_2025_2026.csv")

In [3]:
sentiment['Category'].unique()

<StringArray>
['Satisfaction', 'Security', 'Safety', 'Improvements']
Length: 4, dtype: str

In [4]:
#this is for safety score
safety_aggregate = sentiment[sentiment['Score Type'] == 'Average Score (1-10)']


# keep only improvements
improve = sentiment[
    (sentiment["Category"] == "Improvements") &
    (sentiment["Score Type"] == "Percent of Respondents")
].copy()

# aggregate average support
improve_ranked = (
    improve
    .groupby("Score Grain", as_index=False)["Score"]
    .mean()
    .sort_values("Score", ascending=True)
)

fig = px.bar(
    improve_ranked,
    x="Score",
    y="Score Grain",
    orientation="h",
    title="Transit Improvements Riders Want Most",
    labels={
        "Score": "% of Respondents",
        "Score Grain": "Desired Improvement"
    }
)

fig.update_layout(template="plotly_white")

fig.show()

In [5]:
sentiment['Score Type'].unique()

<StringArray>
['Percent of Respondents', 'Average Score (1-10)']
Length: 2, dtype: str

In [6]:
import pandas as pd
import plotly.express as px

sentiment["Month"] = pd.to_datetime(sentiment["Month"])

df_clean = sentiment.copy()

# focus only on comparable percent-based metrics
df_clean = df_clean[df_clean["Score Type"] == "Percent of Respondents"]

ts = (
    df_clean
    .groupby(["Month", "Category", "Measure"], as_index=False)["Score"]
    .mean()
)

fig = px.line(
    ts,
    x="Month",
    y="Score",
    color="Measure",
    facet_col="Category",
    facet_col_wrap=2,
    markers=True,
    title="MTA Rider Survey Trends by Category and Measure",
    labels={"Score": "% of Respondents"}
)

fig.update_layout(
    hovermode="x unified",
    template="plotly_white"
)

fig.show()

In [7]:
df_clean.head()

,Month,Time Period,Mode,Bus Type,Question Text,Category,Measure,Score Grain,Score Type,Score,Sample Size
0,2026-03-01,NaN,Subway,NaN,Thinking about your experience in [INSERT SUBW...,Satisfaction,Overall Station,Dissatisfied,Percent of Respondents,23.193733,"2,147"
1,2026-03-01,20:00 - 06:29,Subway,NaN,How satisfied were you with your overall subwa...,Satisfaction,Overall Last Trip,Total Satisfied (Very Satisfied + Satisfied),Percent of Respondents,64.641679,321
2,2026-03-01,20:00 - 06:29,Subway,NaN,Thinking about your experience with [INSERT SU...,Satisfaction,Crowding on Trains,Total Satisfied (Very Satisfied + Satisfied),Percent of Respondents,66.773006,321
3,2026-03-01,15:30 - 19:59,Subway,NaN,Thinking about your experience with [INSERT SU...,Satisfaction,Wait Time,Total Satisfied (Very Satisfied + Satisfied),Percent of Respondents,72.775817,548
4,2026-03-01,15:30 - 19:59,Subway,NaN,How satisfied were you with your overall subwa...,Satisfaction,Overall Last Trip,Total Satisfied (Very Satisfied + Satisfied),Percent of Respondents,69.727848,548


In [8]:
df_sat = sentiment[
    (sentiment["Category"] == "Satisfaction") &
    (sentiment["Score Type"] == "Percent of Respondents")
]

ts_sat = (
    df_sat
    .groupby(["Month", "Measure"], as_index=False)["Score"]
    .mean()
)

fig = px.line(
    ts_sat,
    x="Month",
    y="Score",
    color="Measure",
    markers=True,
    title="Rider Satisfaction Over Time (MTA Pulse Survey)"
)

fig.update_layout(
    hovermode="x unified",
    template="plotly_white"
)

fig.show()

In [12]:
mta_hourly = pd.read_csv("data/MTA_Subway_Hourly_Ridership__April30_2026.csv")
mta_hourly['fare_class_category'].unique()

fare_percent = (
    mta_hourly['fare_class_category']
    .value_counts(normalize=True, dropna=False)
    .mul(100)
    .reset_index()
)

fare_percent.columns = ['fare_class_category', 'percent']

fig = px.bar(
    fare_percent,
    x='fare_class_category',
    y='percent',
    title='Percentage Distribution of Fare Class Categories',
    labels={
        'fare_class_category': 'Fare Class Category',
        'percent': 'Percent'
    }
)

fig.update_layout(
    xaxis_tickangle=-45,
    template='plotly_white'
)

fig.show()

### Exploring route data

Route data comes from two subway GTFS feeds:
- Regular GTFS: Standard subway schedules without most temporary service changes and is updated a few times a year

- Supplemented GTFS: most, but not all service changes for the next 7 calendar days. The simpler the service change, the more likely it will not be included. It is updated **hourly**

#### Regular GTFS data exploration

In [ ]:
import pandas as pd
import networkx as nx
import json
import os

os.makedirs("outputs", exist_ok=True)

stops = pd.read_csv("data/standard_route_data/stops.txt")
routes = pd.read_csv("data/standard_route_data/routes.txt")
trips = pd.read_csv("data/standard_route_data/trips.txt")
stop_times = pd.read_csv("data/standard_route_data/stop_times.txt")

#station map
stop_to_station = stops.set_index("stop_id")["parent_station"].to_dict()

#attach route info
stop_times = stop_times.merge(
    trips[["trip_id", "route_id"]],
    on="trip_id",
    how="left"
)
#ordered trip edges
stop_times = stop_times.sort_values(["trip_id", "stop_sequence"])

stop_times["next_stop"] = stop_times.groupby("trip_id")["stop_id"].shift(-1)

edges = stop_times.dropna(subset=["next_stop"])[
    ["stop_id", "next_stop", "trip_id", "route_id"]
]


#convert to station level
edges["from_station"] = edges["stop_id"].map(stop_to_station)
edges["to_station"] = edges["next_stop"].map(stop_to_station)

edges = edges.dropna(subset=["from_station", "to_station"])

# remove self-loops
edges = edges[edges["from_station"] != edges["to_station"]]


#build station table 
stations = stops[[
    "stop_id",
    "stop_name",
    "stop_lat",
    "stop_lon",
    "parent_station"
]].dropna(subset=["parent_station"])

stations = stations.groupby("parent_station").agg({
    "stop_lat": "mean",
    "stop_lon": "mean",
    "stop_name": "first"
}).reset_index()

stations = stations.rename(columns={
    "parent_station": "station_id"
})


#build graph
G = nx.Graph()

for _, r in edges.iterrows():
    G.add_edge(
        r["from_station"],
        r["to_station"],
        weight=1,
        trip_id=r["trip_id"],
        route_id=r["route_id"]   
    )

print("Stations:", len(G.nodes))
print("Edges:", len(G.edges))


#build geo table
stations_small = stations[[
    "station_id",
    "stop_name",
    "stop_lat",
    "stop_lon"
]]

# FROM SIDE
edges_geo = edges.merge(
    stations_small,
    left_on="from_station",
    right_on="station_id"
)

edges_geo = edges_geo.rename(columns={
    "stop_lat": "lat_from",
    "stop_lon": "lon_from"
}).drop(columns=["station_id"])


# TO SIDE
edges_geo = edges_geo.merge(
    stations_small,
    left_on="to_station",
    right_on="station_id"
)

edges_geo = edges_geo.rename(columns={
    "stop_lat": "lat_to",
    "stop_lon": "lon_to"
}).drop(columns=["station_id"])


# EXPORT LINES GEOJSON 
line_features = []

for _, r in edges_geo.iterrows():
    line_features.append({
        "type": "Feature",
        "geometry": {
            "type": "LineString",
            "coordinates": [
                [r["lon_from"], r["lat_from"]],
                [r["lon_to"], r["lat_to"]]
            ]
        },
        "properties": {
            "trip_id": r["trip_id"],
            "route_id": r["route_id"]   
        }
    })

lines_geojson = {
    "type": "FeatureCollection",
    "features": line_features
}

with open("outputs/lines.geojson", "w") as f:
    json.dump(lines_geojson, f)


# EXPORT STATIONS GEOJSON
station_features = []

for _, r in stations.iterrows():
    station_features.append({
        "type": "Feature",
        "geometry": {
            "type": "Point",
            "coordinates": [r["stop_lon"], r["stop_lat"]]
        },
        "properties": {
            "station_id": r["station_id"],
            "name": r["stop_name"]
        }
    })

stations_geojson = {
    "type": "FeatureCollection",
    "features": station_features
}

with open("outputs/stations.geojson", "w") as f:
    json.dump(stations_geojson, f)



print("Final stations:", len(G.nodes))
print("Final connections:", len(G.edges))
print("Sample route_id:", edges_geo["route_id"].iloc[0])

Stations: 496
Edges: 578
Final stations: 496
Final connections: 578
Sample route_id: 1


In [20]:
stops.head()

,stop_id,stop_name,stop_lat,stop_lon,location_type,parent_station
0,101,Van Cortlandt Park-242 St,40.889248,-73.898583,1.0,NaN
1,101N,Van Cortlandt Park-242 St,40.889248,-73.898583,NaN,101
2,101S,Van Cortlandt Park-242 St,40.889248,-73.898583,NaN,101
3,103,238 St,40.884667,-73.900870,1.0,NaN
4,103N,238 St,40.884667,-73.900870,NaN,103


In [21]:
trips.head()

,route_id,trip_id,service_id,trip_headsign,direction_id,shape_id
0,1,ASP26GEN-1038-Sunday-00_000600_1..S03R,Sunday,South Ferry,1,1..S03R
1,1,ASP26GEN-1038-Sunday-00_002600_1..S03R,Sunday,South Ferry,1,1..S03R
2,1,ASP26GEN-1038-Sunday-00_004600_1..S03R,Sunday,South Ferry,1,1..S03R
3,1,ASP26GEN-1038-Sunday-00_006600_1..S03R,Sunday,South Ferry,1,1..S03R
4,1,ASP26GEN-1038-Sunday-00_007200_1..N03R,Sunday,Van Cortlandt Park-242 St,0,1..N03R


In [22]:
stop_times.head()

,trip_id,stop_id,arrival_time,departure_time,stop_sequence,next_stop
0,ASP26GEN-1038-Sunday-00_000600_1..S03R,101S,00:06:00,00:06:00,1,103S
1,ASP26GEN-1038-Sunday-00_000600_1..S03R,103S,00:07:30,00:07:30,2,104S
2,ASP26GEN-1038-Sunday-00_000600_1..S03R,104S,00:09:00,00:09:00,3,106S
3,ASP26GEN-1038-Sunday-00_000600_1..S03R,106S,00:10:30,00:10:30,4,107S
4,ASP26GEN-1038-Sunday-00_000600_1..S03R,107S,00:12:00,00:12:00,5,108S


In [23]:
edges.head()

,stop_id,next_stop,trip_id
0,101S,103S,ASP26GEN-1038-Sunday-00_000600_1..S03R
1,103S,104S,ASP26GEN-1038-Sunday-00_000600_1..S03R
2,104S,106S,ASP26GEN-1038-Sunday-00_000600_1..S03R
3,106S,107S,ASP26GEN-1038-Sunday-00_000600_1..S03R
4,107S,108S,ASP26GEN-1038-Sunday-00_000600_1..S03R
